In [1]:
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pytorch_tcn import TCN

class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        # output size = vocab size
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]
        output = self.fc(lstm_out)
        return output


In [2]:
LSTM_model = LSTMModel(
    vocab_size=3040,
    embedding_dim=128,
    hidden_dim=256
)
LSTM_model.load_state_dict(
    torch.load("lstm_next_word.pth")
)

LSTM_model.eval()

LSTMModel(
  (embedding): Embedding(3040, 128)
  (lstm): LSTM(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=3040, bias=True)
)

In [3]:
class TCNNextWord(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.tcn = TCN(
            num_inputs=128,
            num_channels=[128, 128, 128],
            kernel_size=3,
            dropout=0.2
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):

        # x shape:
        # (batch_size, seq_len)

        x = self.embedding(x)

        # TCN expects:
        # (batch, channels, seq_len)

        x = x.permute(0, 2, 1)

        x = self.tcn(x)

        # last timestep

        x = x[:, :, -1]

        out = self.fc(x)

        return out

In [4]:
vocab_size = 3040
TCN_model = TCNNextWord(vocab_size)

In [5]:
TCN_model.load_state_dict(
    torch.load("TCN_next_word.pth")
)
TCN_model.eval()

TCNNextWord(
  (embedding): Embedding(3040, 128, padding_idx=0)
  (tcn): TCN(
    (network): ModuleList(
      (0): TemporalBlock(
        (conv1): ParametrizedTemporalConv1d(
          128, 128, kernel_size=(3,), stride=(1,)
          (padder): TemporalPad1d(
            (pad): ConstantPad1d(padding=(2, 0), value=0.0)
          )
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
        (conv2): ParametrizedTemporalConv1d(
          128, 128, kernel_size=(3,), stride=(1,)
          (padder): TemporalPad1d(
            (pad): ConstantPad1d(padding=(2, 0), value=0.0)
          )
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
        (activation1): ReLU()
        (activation2): ReLU()
        (activation_final): ReLU()
        (dropout1): Dropout(p=0.2, inplace=False)
        (dro

In [6]:
import json
import torch

with open('word_to_id.json', 'r') as f:
    id = json.load(f)

with open('id_to_word.json', 'r') as f:
    word_id = json.load(f)

In [7]:
df = pd.read_csv("final_data.csv")
df['text'] = df['text'].apply(ast.literal_eval)
df['next_word'] = df['next_word'].apply(ast.literal_eval)
df.head()

,text,next_word
0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[1106]
1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[59]
2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[2980]
3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[2952]
4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 754...",[1916]


In [8]:
test_df = df.sample(100)
x = test_df['text'].values
y = test_df['next_word'].values

x = [torch.tensor(i, dtype=torch.long) for i in x]
y = [torch.tensor(i, dtype=torch.long) for i in y]

In [9]:
acc_lstm = 0
acc_tcn = 0
for i in range(len(x)):
    seq = x[i].unsqueeze(0)

    with torch.no_grad():

        lstm_pred_idx = LSTM_model(seq)
        tcn_pred_idx = TCN_model(seq)
        lstm_pred_idx = torch.argmax(lstm_pred_idx, dim=1).item()
        tcn_pred_idx = torch.argmax(tcn_pred_idx, dim=1).item() 

    true_idx = y[i].item()

    if lstm_pred_idx == true_idx:
        acc_lstm += 1

    if tcn_pred_idx == true_idx:
        acc_tcn += 1

print("LSTM Accuracy:", (acc_lstm/len(x))*100, "%")
print("TCN Accuracy:", (acc_tcn/len(x))*100, "%")

LSTM Accuracy: 66.0 %
TCN Accuracy: 36.0 %
